# Phase 1 — OpenML Data Pull

Pulls 60–70 tabular classification datasets from OpenML, caches them locally, and
saves a manifest CSV listing each dataset's key properties.

**Filters** (from CLAUDE.md):
- 100 ≤ n_instances ≤ 100,000
- 2 ≤ n_classes ≤ 10
- n_features < 200
- No missing values
- Tabular / classification tasks only

**Output**: `data/meta_table/dataset_manifest.csv`

**Showcase datasets are excluded** from the manifest to prevent training leakage.

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import openml

# ── Paths ────────────────────────────────────────────────────────────────────
ROOT        = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DIR     = os.path.join(ROOT, 'data', 'raw')
META_DIR    = os.path.join(ROOT, 'data', 'meta_table')
MANIFEST    = os.path.join(META_DIR, 'dataset_manifest.csv')

os.makedirs(RAW_DIR,  exist_ok=True)
os.makedirs(META_DIR, exist_ok=True)

# Cache OpenML downloads to data/raw/
openml.config.cache_directory = RAW_DIR

print(f'ROOT     : {ROOT}')
print(f'RAW_DIR  : {RAW_DIR}')
print(f'MANIFEST : {MANIFEST}')

ROOT     : c:\MLResearch
RAW_DIR  : c:\MLResearch\data\raw
MANIFEST : c:\MLResearch\data\meta_table\dataset_manifest.csv


In [3]:
# ── Showcase dataset IDs — NEVER include these in meta-training ───────────────
# Identified by OpenML dataset ID (authoritative — names can alias)
SHOWCASE_IDS = {
    61,    # iris
    187,   # wine
    15,    # breast-w (Breast Cancer Wisconsin)
    53,    # heart-statlog (Heart Disease UCI)
    40966, # penguins (Palmer Penguins)
    37,    # diabetes (Pima)
    54,    # vehicle (Vehicle Silhouettes)
    1590,  # adult (Adult Income)
    1597,  # creditcard (Credit Card Fraud)
}

print(f'Showcase IDs excluded from training: {sorted(SHOWCASE_IDS)}')

Showcase IDs excluded from training: [15, 37, 53, 54, 61, 187, 1590, 1597, 40966]


In [4]:
# ── Filter constants (from CLAUDE.md) ────────────────────────────────────────
MIN_INSTANCES  = 100
MAX_INSTANCES  = 100_000
MIN_CLASSES    = 3      # was 2 — exclude binary to focus on multi-class
MAX_CLASSES    = 10
MAX_FEATURES   = 200
TARGET_COUNT   = 120    # was 70 — larger pool for better meta-learner coverage

print('Filters:')
print(f'  instances : {MIN_INSTANCES:,} – {MAX_INSTANCES:,}')
print(f'  classes   : {MIN_CLASSES} – {MAX_CLASSES}')
print(f'  features  : < {MAX_FEATURES}')
print(f'  missing   : none allowed')
print(f'  target    : {TARGET_COUNT} datasets')

Filters:
  instances : 100 – 100,000
  classes   : 3 – 10
  features  : < 200
  missing   : none allowed
  target    : 120 datasets


In [ ]:
# ── Fetch OpenML task list (supervised classification) ────────────────────────
# Task type 1 = Supervised Classification on OpenML
print('Fetching task list from OpenML (this may take ~30 s on first run)...')

tasks = openml.tasks.list_tasks(
    task_type=openml.tasks.TaskType.SUPERVISED_CLASSIFICATION,
    output_format='dataframe',
)

print(f'Total tasks returned: {len(tasks):,}')
tasks.head(10)

Fetching task list from OpenML (this may take ~30 s on first run)...


In [6]:
# ── Inspect available columns for filtering ───────────────────────────────────
print('Columns:', tasks.columns.tolist())
print()
# Show which size/quality columns are present
for col in ['NumberOfInstances', 'NumberOfFeatures', 'NumberOfClasses',
            'NumberOfMissingValues', 'did', 'name']:
    present = col in tasks.columns
    print(f'  {col:35s} present={present}')

Columns: ['tid', 'ttid', 'did', 'name', 'task_type', 'status', 'estimation_procedure', 'evaluation_measures', 'source_data', 'target_feature', 'MajorityClassSize', 'MaxNominalAttDistinctValues', 'MinorityClassSize', 'NumberOfClasses', 'NumberOfFeatures', 'NumberOfInstances', 'NumberOfInstancesWithMissingValues', 'NumberOfMissingValues', 'NumberOfNumericFeatures', 'NumberOfSymbolicFeatures', 'cost_matrix']

  NumberOfInstances                   present=True
  NumberOfFeatures                    present=True
  NumberOfClasses                     present=True
  NumberOfMissingValues               present=True
  did                                 present=True
  name                                present=True


In [7]:
# ── Apply size / quality filters ──────────────────────────────────────────────
df = tasks.copy()

# Drop rows with missing filter columns
needed = ['NumberOfInstances', 'NumberOfFeatures', 'NumberOfClasses',
          'NumberOfMissingValues', 'did']
df = df.dropna(subset=needed)

# Cast to numeric
for col in needed[:-1]:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna(subset=needed)

before = len(df)
df = df[
    (df['NumberOfInstances']     >= MIN_INSTANCES) &
    (df['NumberOfInstances']     <= MAX_INSTANCES) &
    (df['NumberOfClasses']       >= MIN_CLASSES)   &
    (df['NumberOfClasses']       <= MAX_CLASSES)   &
    (df['NumberOfFeatures']      <  MAX_FEATURES)  &
    (df['NumberOfMissingValues'] == 0)
]
print(f'After size/quality filters: {len(df):,}  (from {before:,})')

After size/quality filters: 2,748  (from 5,520)


In [8]:
# ── Deduplicate by dataset ID, exclude showcase IDs ──────────────────────────
df = df.drop_duplicates(subset='did')
df = df[~df['did'].isin(SHOWCASE_IDS)]

# Sort by instance count for diversity; reset index
df = df.sort_values('NumberOfInstances').reset_index(drop=True)

print(f'After dedup + showcase exclusion: {len(df):,} candidate datasets')

After dedup + showcase exclusion: 1,312 candidate datasets


In [10]:
# ── Select a diverse subset of TARGET_COUNT datasets ──────────────────────────
# Stratify by: class-count bin × instance-size bin × imbalance bin
# Guarantees representation across easy (balanced, few classes) and hard
# (imbalanced, many classes) cases — avoids the binary-heavy skew of pure
# size-based sampling.

import math

# ─── Compute stratification axes ─────────────────────────────────────────────
df['log_n'] = np.log10(df['NumberOfInstances'])

# Imbalance ratio: MinorityClassSize / MajorityClassSize (1.0 = perfectly balanced)
df['imbalance_ratio'] = (
    df['MinorityClassSize'] / df['MajorityClassSize']
).clip(0, 1)
df = df.dropna(subset=['imbalance_ratio'])   # drop rows missing size info

# class bin  — 3–5 classes (0) vs 6–10 classes (1)
df['class_bin']   = (df['NumberOfClasses'] >= 6).astype(int)

# size bin   — 3 equal-width log-instance bins
df['size_bin']    = pd.cut(df['log_n'], bins=3, labels=False)

# balance bin — balanced (ratio > 0.5, label 1) vs imbalanced (ratio ≤ 0.5, label 0)
df['balance_bin'] = (df['imbalance_ratio'] > 0.5).astype(int)

df['stratum'] = (df['class_bin'].astype(str) + '_'
               + df['size_bin'].astype(str)  + '_'
               + df['balance_bin'].astype(str))

strata_counts = df.groupby('stratum').size().sort_index()
print('Strata  (class_bin _ size_bin _ balance_bin):')
print('  class_bin : 0 = 3–5 classes,  1 = 6–10 classes')
print('  balance_bin: 0 = imbalanced (ratio ≤ 0.5),  1 = balanced (ratio > 0.5)')
print()
print(strata_counts.to_string())
print(f'\nTotal available after all filters: {len(df):,}')

# ─── Sample proportionally from each non-empty stratum ───────────────────────
non_empty = strata_counts[strata_counts > 0]
n_strata  = len(non_empty)
per_strat = math.ceil(TARGET_COUNT / n_strata)

sampled = (
    df.groupby('stratum', group_keys=False)
      .apply(lambda g: g.sample(min(per_strat, len(g)), random_state=42))
)

# Shuffle then trim to exactly TARGET_COUNT
sampled = sampled.sample(frac=1, random_state=42).head(TARGET_COUNT).reset_index(drop=True)

# Recompute stratum after groupby (pandas 2.x drops the groupby key column)
sampled['stratum'] = (sampled['class_bin'].astype(str) + '_'
                    + sampled['size_bin'].astype(str)  + '_'
                    + sampled['balance_bin'].astype(str))

print(f'\nSelected {len(sampled)} datasets')
print('\nStratum breakdown of selected:')
print(sampled['stratum'].value_counts().sort_index().to_string())
print()
print(sampled[['did', 'name', 'NumberOfInstances', 'NumberOfFeatures',
               'NumberOfClasses', 'imbalance_ratio']].to_string(index=False))

Strata  (class_bin _ size_bin _ balance_bin):
  class_bin : 0 = 3–5 classes,  1 = 6–10 classes
  balance_bin: 0 = imbalanced (ratio ≤ 0.5),  1 = balanced (ratio > 0.5)

stratum
0_0_0    39
0_0_1    23
0_1_0    77
0_1_1    29
0_2_0    28
0_2_1     4
1_0_0    11
1_0_1     6
1_1_0    67
1_1_1    37
1_2_0     9
1_2_1     5

Total available after all filters: 335

Selected 104 datasets

Stratum breakdown of selected:
stratum
0_0_0    10
0_0_1    10
0_1_0    10
0_1_1    10
0_2_0    10
0_2_1     4
1_0_0    10
1_0_1     6
1_1_0    10
1_1_1    10
1_2_0     9
1_2_1     5

  did                                                                              name  NumberOfInstances  NumberOfFeatures  NumberOfClasses  imbalance_ratio
41004                                           jungle_chess_2pcs_endgame_lion_elephant             4704.0              47.0              3.0         0.584897
 4153                                  Smartphone-Based_Recognition_of_Human_Activities              180.0       

In [11]:
# ── Download & verify each dataset ───────────────────────────────────────────
# Downloads are cached by the OpenML library under RAW_DIR.
# We verify that each dataset loads correctly and record actual row/col counts.

records = []
failed  = []

for i, row in sampled.iterrows():
    did  = int(row['did'])
    name = row.get('name', '')
    try:
        ds     = openml.datasets.get_dataset(
                     did,
                     download_data=True,
                     download_qualities=True,
                     download_features_meta_data=False,
                 )
        X, y, _, _  = ds.get_data(dataset_format='dataframe',
                                   target=ds.default_target_attribute)

        # Verify no missing values after load
        if X.isnull().any().any() or (y is not None and y.isnull().any()):
            failed.append((did, name, 'missing values after load'))
            continue

        n_inst    = X.shape[0]
        n_feat    = X.shape[1]
        n_classes = int(y.nunique()) if y is not None else 0

        records.append({
            'dataset_id' : did,
            'name'       : ds.name,
            'n_instances': n_inst,
            'n_features' : n_feat,
            'n_classes'  : n_classes,
        })

        print(f'[{i+1:3d}/{len(sampled)}] OK  id={did:6d}  {ds.name[:40]:40s}  '
              f'{n_inst:6d}r × {n_feat:3d}c  {n_classes}cls')

    except Exception as e:
        failed.append((did, name, str(e)))
        print(f'[{i+1:3d}/{len(sampled)}] FAIL id={did}  {name}  — {e}')

print(f'\nLoaded: {len(records)}  Failed: {len(failed)}')

[  1/104] OK  id= 41004  jungle_chess_2pcs_endgame_lion_elephant     4704r ×  46c  3cls
[  2/104] OK  id=  4153  Smartphone-Based_Recognition_of_Human_Ac     180r ×  66c  6cls
[  3/104] OK  id=  1465  breast-tissue                                106r ×   9c  6cls
[  4/104] OK  id=   119  BNG(cmc,nominal,55296)                     55296r ×   9c  3cls
[  5/104] OK  id=    26  nursery                                    12960r ×   8c  5cls
[  6/104] OK  id=   373  UNIX_user_data                              9100r ×   2c  9cls
[  7/104] OK  id=   375  JapaneseVowels                              9961r ×  14c  9cls
[  8/104] FAIL id=46978  internet_firewall  — HTTPSConnectionPool(host='data.openml.org', port=443): Read timed out.
[  9/104] OK  id= 46879  mental_health_detection                      540r ×  14c  4cls
[ 10/104] OK  id= 47000  QSAR_Bioconcentration_classification         779r ×  12c  3cls
[ 11/104] OK  id= 42186  JuanFeldmanIris                              150r ×   4c  3cls
[ 1

[103/104] OK  id= 45927  Products                                   73503r ×   3c  5cls
[104/104] OK  id=    32  pendigits                                  10992r ×  16c  10cls

Loaded: 103  Failed: 1


In [12]:
# ── Save manifest ─────────────────────────────────────────────────────────────
manifest = pd.DataFrame(records)
manifest.to_csv(MANIFEST, index=False)

print(f'Manifest saved → {MANIFEST}')
print(f'Shape: {manifest.shape}')
manifest

Manifest saved → c:\MLResearch\data\meta_table\dataset_manifest.csv
Shape: (103, 5)


,dataset_id,name,n_instances,n_features,n_classes
0,41004,jungle_chess_2pcs_endgame_lion_elephant,4704,46,3
1,4153,Smartphone-Based_Recognition_of_Human_Activities,180,66,6
2,1465,breast-tissue,106,9,6
3,119,"BNG(cmc,nominal,55296)",55296,9,3
4,26,nursery,12960,8,5
...,...,...,...,...,...
98,44469,yeast_seed_1_nrows_2000_nclasses_10_ncols_100_...,1484,8,10
99,44554,vehicle_seed_1_nrows_2000_nclasses_10_ncols_10...,846,18,4
100,45714,PriceRunner,35300,5,10
101,45927,Products,73503,3,5


In [13]:
# ── Sanity checks ─────────────────────────────────────────────────────────────

# 1. No showcase IDs must appear in the manifest
leaked = set(manifest['dataset_id']) & SHOWCASE_IDS
assert len(leaked) == 0, f'SHOWCASE LEAK: {leaked}'
print('✓ No showcase IDs in manifest')

# 2. All instances within allowed range
assert manifest['n_instances'].between(MIN_INSTANCES, MAX_INSTANCES).all(), \
    'Instance count out of range'
print('✓ All datasets within instance range')

# 3. All class counts within allowed range
assert manifest['n_classes'].between(MIN_CLASSES, MAX_CLASSES).all(), \
    'Class count out of range'
print('✓ All datasets within class range')

# 4. All feature counts below max
assert (manifest['n_features'] < MAX_FEATURES).all(), \
    'Feature count out of range'
print('✓ All datasets below feature limit')

# 5. No duplicate IDs
assert manifest['dataset_id'].is_unique, 'Duplicate dataset IDs'
print('✓ No duplicate dataset IDs')

# 6. Count
print(f'\nFinal manifest: {len(manifest)} datasets ready for Phase 2 (LSE computation)')

✓ No showcase IDs in manifest
✓ All datasets within instance range
✓ All datasets within class range
✓ All datasets below feature limit
✓ No duplicate dataset IDs

Final manifest: 103 datasets ready for Phase 2 (LSE computation)


In [14]:
# ── Summary statistics ────────────────────────────────────────────────────────
print('=== Dataset manifest summary ===')
print(manifest[['n_instances','n_features','n_classes']].describe().round(1).to_string())

print('\n=== Class distribution ===')
print(manifest['n_classes'].value_counts().sort_index())

if failed:
    print('\n=== Failed downloads ===')
    for did, name, reason in failed:
        print(f'  id={did}  {name}  — {reason}')

=== Dataset manifest summary ===
       n_instances  n_features  n_classes
count        103.0       103.0      103.0
mean        8780.0        27.8        5.8
std        17019.2        34.7        2.4
min          101.0         1.0        3.0
25%          570.0         8.0        4.0
50%         2000.0        12.0        5.0
75%         9136.0        32.5        7.0
max        78053.0       180.0       10.0

=== Class distribution ===
n_classes
3     25
4     12
5     16
6     19
7      7
8      4
9      5
10    15
Name: count, dtype: int64

=== Failed downloads ===
  id=46978  internet_firewall  — HTTPSConnectionPool(host='data.openml.org', port=443): Read timed out.


In [15]:
# ── Replace failed dataset 350 (webdata_wXa — malformed target column) ────────
# Candidates tried in order; first successful download wins.
REPLACEMENT_CANDIDATES = [
    32,   # satimage   — 6430r × 36f, 6cls
    24,   # mushroom   — 8124r × 22f, 2cls
    29,   # page-blocks — 5473r × 10f, 5cls
    28,   # optdigits  — 5620r × 64f, 10cls
]

import pandas as pd
import os

MANIFEST = os.path.join(os.path.abspath(os.path.join(os.getcwd(), '..')),
                        'data', 'meta_table', 'dataset_manifest.csv')
SHOWCASE_IDS = {
    61, 187, 15, 53, 40966, 37, 54, 1590, 1597,
}

manifest = pd.read_csv(MANIFEST)
existing_ids = set(manifest['dataset_id'])

replacement = None
for cand_id in REPLACEMENT_CANDIDATES:
    if cand_id in existing_ids or cand_id in SHOWCASE_IDS:
        print(f'  Skip {cand_id} — already in manifest or showcase')
        continue
    try:
        import openml
        ds = openml.datasets.get_dataset(
                 cand_id,
                 download_data=True,
                 download_qualities=True,
                 download_features_meta_data=False,
             )
        X, y, _, _ = ds.get_data(dataset_format='dataframe',
                                  target=ds.default_target_attribute)
        if X.isnull().any().any() or (y is not None and y.isnull().any()):
            print(f'  Skip {cand_id} ({ds.name}) — missing values after load')
            continue
        n_inst    = X.shape[0]
        n_feat    = X.shape[1]
        n_classes = int(y.nunique()) if y is not None else 0
        # Verify filters
        if not (100 <= n_inst <= 100_000 and 2 <= n_classes <= 10 and n_feat < 200):
            print(f'  Skip {cand_id} ({ds.name}) — fails filter: '
                  f'{n_inst}r {n_feat}f {n_classes}cls')
            continue
        replacement = {
            'dataset_id' : cand_id,
            'name'       : ds.name,
            'n_instances': n_inst,
            'n_features' : n_feat,
            'n_classes'  : n_classes,
        }
        print(f'Replacement: id={cand_id}  {ds.name}  {n_inst}r × {n_feat}f  {n_classes}cls')
        break
    except Exception as e:
        print(f'  Skip {cand_id} — {e}')

assert replacement is not None, 'All replacement candidates failed — add more to the list'

manifest = pd.concat([manifest, pd.DataFrame([replacement])], ignore_index=True)
manifest.to_csv(MANIFEST, index=False)

print(f'Manifest updated: {len(manifest)} datasets total')
assert replacement['dataset_id'] not in SHOWCASE_IDS, 'Showcase leak!'
assert manifest['dataset_id'].is_unique, 'Duplicate IDs!'
print('Sanity checks passed.')


  Skip 32 — already in manifest or showcase
  Skip 24 (mushroom) — missing values after load
  Skip 29 (credit-approval) — missing values after load
Replacement: id=28  optdigits  5620r × 64f  10cls
Manifest updated: 104 datasets total
Sanity checks passed.
